# Sentinel Job — Anomaly & Schema-Drift Watchdog (module M5, PDF task 5.5)

Databricks Job wrapper around the same three read-only checks the CLI
(`scripts/sentinel.py`) runs locally: `sql/sentinel/daily_event_volumes.sql`,
`sql/sentinel/daily_funnel_rates.sql`, `sql/sentinel/schema_snapshot.sql`.
Both entry points share the exact same band-math / registry-diff logic —
`src/agent/sentinel_core.py` — imported here when the workspace layout
allows it (this notebook checked out via **Databricks Repos**, so
`<repo>/src` is a real path on the driver), with a clearly-labelled,
self-contained fallback if it is not.

**Design philosophy** (see `docs/sentinel_design.md`): STATISTICS DETECT —
SQL window functions and registry diffing produce every finding below. The
LLM, if one is configured, only NARRATES those findings in prose; it never
invents a number. The report this notebook writes is a **DRAFT — pending
analyst approval**: this Job never sends anything anywhere on its own.

**Trigger**: intended to run as a Databricks Workflows Job, scheduled daily
shortly after the ETL/medallion job finishes (e.g. 06:00) — see
`docs/sentinel_design.md` and `docs/deploy_guide.md` (Turkish, Job-creation
steps).

In [ ]:
# Widgets -- every value a Job schedule or a manual run might want to
# override. Safe to leave every default as-is for a normal scheduled run.
dbutils.widgets.text("catalog", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("schema", "sonova", "Schema (matches DATABRICKS_SCHEMA)")
dbutils.widgets.text("as_of", "", "as-of date YYYY-MM-DD (blank = auto: matured max date)")
dbutils.widgets.text(
    "repo_root",
    "/Workspace/Repos/sonova_case",
    "Path this repo is checked out at (Databricks Repos) -- used to import"
    " agent.sentinel_core and to read sql/sentinel/*.sql; leave the default"
    " unless your Repos folder name differs.",
)

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
as_of_override = dbutils.widgets.get("as_of").strip() or None
repo_root = dbutils.widgets.get("repo_root").strip()

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

## 1. Load the detection core: real import first, documented fallback second

`sentinel_core` is `src/agent/sentinel_core.py`, imported unmodified when
this notebook's `repo_root` widget points at a real Databricks Repos clone
of this repository (the normal case for a Job notebook task that lives in
the repo). If that import fails -- e.g. the notebook was copied into the
workspace on its own, without the rest of the repo -- the fallback cell
below defines a **mirror** of just the functions this notebook calls, with
a header comment saying so. Keep that fallback in sync with
`src/agent/sentinel_core.py` if the real module's contract ever changes;
the import path is always tried first specifically so this fallback is
the exception, not the normal path.

In [ ]:
import sys

sentinel_core_imported = False
try:
    sys.path.insert(0, f"{repo_root}/src")
    from agent import sentinel_core as sc  # noqa: F401
    from agent.llm import get_llm  # noqa: F401

    sentinel_core_imported = True
    print(f"Imported agent.sentinel_core from {repo_root}/src (real import).")
except ImportError as exc:
    print(
        f"Could not import agent.sentinel_core from {repo_root}/src ({exc}).\n"
        "Falling back to the mirrored subset defined in the next cell -- see"
        " that cell's header comment."
    )

In [ ]:
# NOTE: this cell only runs its body if the real import above failed. It
# MIRRORS src/agent/sentinel_core.py -- specifically the subset this
# notebook needs (SQL templating, the band-severity rule, the four
# registry-diff checks, and the deterministic narration template). Keep it
# in sync with that file if its contract changes; the import cell above is
# always tried first so this path is only exercised when the workspace
# genuinely cannot see the repo's src/ tree.
import datetime as dt
import json
import re
from dataclasses import dataclass, field
from typing import Any, Optional

import pandas as pd

if not sentinel_core_imported:

    class sc:  # noqa: N801 -- namespaced like a module on purpose
        SEVERITIES = ("info", "warning", "critical")
        _SEVERITY_RANK = {s: i for i, s in enumerate(SEVERITIES)}
        EVENT_MATURITY_BUFFER_DAYS = 6
        DEFAULT_THRESHOLDS = {
            "band_multiplier_info": 1.5,
            "band_multiplier_warning": 2.5,
            "band_multiplier_critical": 4.0,
            "min_volume_floor": 5,
            "min_history_days": 14,
        }
        _TABLE_SOURCE = {"web_events": "web", "app_events": "app"}
        _AS_OF_TOKEN = "{{as_of}}"
        _CHECK_MARKER = re.compile(r"^--\s*@check:\s*(\w+)\s*$")

        @dataclass
        class Finding:
            check: str
            severity: str
            subject: str
            message: str
            details: dict = field(default_factory=dict)

            def to_dict(self):
                return {
                    "check": self.check, "severity": self.severity,
                    "subject": self.subject, "message": self.message,
                    "details": self.details,
                }

        @staticmethod
        def worst_severity(findings):
            if not findings:
                return None
            return max((f.severity for f in findings), key=sc._SEVERITY_RANK.get)

        @staticmethod
        def exit_code(findings):
            worst = sc.worst_severity(findings)
            return {None: 0, "info": 0, "warning": 1, "critical": 2}[worst]

        @staticmethod
        def sort_findings(findings):
            return sorted(findings, key=lambda f: (-sc._SEVERITY_RANK[f.severity], f.check, f.subject))

        @staticmethod
        def render_as_of(sql_text, as_of):
            return sql_text.replace(sc._AS_OF_TOKEN, as_of)

        @staticmethod
        def load_named_statements(sql_text):
            statements, current_name, current_lines = {}, None, []

            def flush():
                nonlocal current_name, current_lines
                if current_name is not None:
                    body = "\n".join(current_lines).strip()
                    if body.endswith(";"):
                        body = body[:-1].strip()
                    if body:
                        statements[current_name] = body
                current_name, current_lines = None, []

            for line in sql_text.splitlines():
                marker = sc._CHECK_MARKER.match(line.strip())
                if marker:
                    flush()
                    current_name = marker.group(1)
                    continue
                if line.lstrip().startswith("--"):
                    continue
                if current_name is not None:
                    current_lines.append(line)
            flush()
            return statements

        @staticmethod
        def default_as_of(driver):
            df = driver.query(
                "SELECT MAX(d) AS max_day FROM ("
                "  SELECT CAST(MAX(event_timestamp) AS DATE) AS d FROM web_events"
                "  UNION ALL"
                "  SELECT CAST(MAX(event_timestamp) AS DATE) AS d FROM app_events"
                ") AS horizons"
            )
            max_day = pd.Timestamp(df["max_day"].iloc[0]).date()
            return (max_day - dt.timedelta(days=sc.EVENT_MATURITY_BUFFER_DAYS)).isoformat()

        @staticmethod
        def load_registry(path):
            registry = json.loads(open(path, encoding="utf-8").read())
            thresholds = dict(sc.DEFAULT_THRESHOLDS)
            thresholds.update(registry.get("thresholds") or {})
            registry["thresholds"] = thresholds
            return registry

        @staticmethod
        def _band_severity(deviation_z, volume_signal, history_days, thresholds):
            if deviation_z is None or pd.isna(deviation_z):
                return None
            if history_days < thresholds["min_history_days"]:
                return None
            if volume_signal < thresholds["min_volume_floor"]:
                return None
            az = abs(deviation_z)
            if az >= thresholds["band_multiplier_critical"]:
                return "critical"
            if az >= thresholds["band_multiplier_warning"]:
                return "warning"
            if az >= thresholds["band_multiplier_info"]:
                return "info"
            return None

        @staticmethod
        def _as_date_str(value):
            return pd.Timestamp(value).date().isoformat()

        @staticmethod
        def evaluate_volume_findings(volume_df, thresholds):
            findings = []
            for row in volume_df.itertuples(index=False):
                day = sc._as_date_str(row.day)
                volume_signal = max(row.actual_count, row.band_avg or 0)
                severity = sc._band_severity(row.deviation_z, volume_signal, row.band_days or 0, thresholds)
                if severity is None:
                    continue
                direction = "above" if row.deviation_z >= 0 else "below"
                findings.append(sc.Finding(
                    check="daily_event_volume", severity=severity,
                    subject=f"{row.source}:{row.event_name}:{row.segment}",
                    message=(
                        f"{row.event_name} ({row.source} / {row.segment}) on {day}: "
                        f"actual={row.actual_count} is {abs(row.deviation_z):.2f}\u03c3 "
                        f"{direction} the trailing 28-day band "
                        f"({row.band_avg:.1f} \u00b1 {row.band_stddev:.1f})."
                    ),
                    details={"day": day, "event_name": row.event_name, "segment": row.segment,
                             "source": row.source, "actual_count": int(row.actual_count),
                             "deviation_z": float(row.deviation_z)},
                ))
            return findings

        @staticmethod
        def evaluate_funnel_rate_findings(rates_df, thresholds):
            findings = []
            for row in rates_df.itertuples(index=False):
                day = sc._as_date_str(row.day)
                severity = sc._band_severity(row.deviation_z, row.from_count, row.band_days or 0, thresholds)
                if severity is None:
                    continue
                direction = "above" if row.deviation_z >= 0 else "below"
                findings.append(sc.Finding(
                    check="daily_funnel_rate", severity=severity, subject=f"rate:{row.step}",
                    message=(
                        f"{row.step} rate on {day}: actual={row.actual_rate:.1%} "
                        f"({row.to_count}/{row.from_count}) is {abs(row.deviation_z):.2f}\u03c3 "
                        f"{direction} the trailing 28-day band "
                        f"({row.band_avg:.1%} \u00b1 {row.band_stddev:.1%})."
                    ),
                    details={"day": day, "step": row.step, "deviation_z": float(row.deviation_z)},
                ))
            return findings

        @staticmethod
        def diff_event_names(event_names_df, registry):
            findings = []
            expected = registry["expected_event_names"]
            actual_by_table = {t: set(g["event_name"]) for t, g in event_names_df.groupby("table_name")}
            for table_name, expected_names in expected.items():
                actual_names = actual_by_table.get(table_name, set())
                for name in sorted(actual_names - set(expected_names)):
                    findings.append(sc.Finding("schema_event_names", "warning", f"event_name:{table_name}.{name}",
                        f"New event_name '{name}' observed in {table_name}, not in the registry.",
                        {"table": table_name, "event_name": name, "kind": "new"}))
                for name in sorted(set(expected_names) - actual_names):
                    findings.append(sc.Finding("schema_event_names", "critical", f"event_name:{table_name}.{name}",
                        f"Expected event_name '{name}' is no longer observed in ANY row of {table_name}.",
                        {"table": table_name, "event_name": name, "kind": "vanished"}))
            return findings

        @staticmethod
        def diff_app_versions(app_versions_df, registry):
            expected = set(registry["expected_app_versions"])
            actual = set(app_versions_df["app_version"])
            return [
                sc.Finding("schema_app_versions", "warning", f"app_version:{v}",
                           f"New app_version '{v}' observed in app_events, not in the registry.",
                           {"app_version": v, "kind": "new"})
                for v in sorted(actual - expected)
            ]

        @staticmethod
        def diff_columns(columns_df, registry):
            findings = []
            expected_schema = registry["expected_schema"]
            actual_by_table = {t: dict(zip(g["column_name"], g["data_type"])) for t, g in columns_df.groupby("table_name")}
            for table_name, expected in expected_schema.items():
                expected_columns = expected["columns"]
                actual_columns = actual_by_table.get(table_name, {})
                for c in sorted(set(actual_columns) - set(expected_columns)):
                    findings.append(sc.Finding("schema_columns", "warning", f"column:{table_name}.{c}",
                        f"New column '{c}' observed on {table_name}, not in the registry.",
                        {"table": table_name, "column": c, "kind": "new"}))
                for c in sorted(set(expected_columns) - set(actual_columns)):
                    findings.append(sc.Finding("schema_columns", "critical", f"column:{table_name}.{c}",
                        f"Expected column '{c}' is MISSING from {table_name}.",
                        {"table": table_name, "column": c, "kind": "missing"}))
                for c in sorted(set(expected_columns) & set(actual_columns)):
                    if expected_columns[c] != actual_columns[c]:
                        findings.append(sc.Finding("schema_columns", "warning", f"column:{table_name}.{c}",
                            f"Column '{table_name}.{c}' changed type: expected {expected_columns[c]}, observed {actual_columns[c]}.",
                            {"table": table_name, "column": c, "kind": "type_changed"}))
            return findings

        @staticmethod
        def check_missing_events_today(volume_df, registry, as_of):
            findings = []
            observed_by_source = ({} if volume_df.empty else
                {s: set(g["event_name"]) for s, g in volume_df.groupby("source")})
            for table_name, expected_names in registry["expected_event_names"].items():
                source = sc._TABLE_SOURCE.get(table_name)
                if source is None:
                    continue
                observed = observed_by_source.get(source, set())
                for name in sorted(set(expected_names) - observed):
                    findings.append(sc.Finding("missing_event_today", "critical", f"missing_today:{table_name}.{name}",
                        f"Expected event '{name}' has ZERO recorded rows in {table_name} on {as_of} (registry expects it daily).",
                        {"table": table_name, "event_name": name, "as_of": as_of}))
            return findings

        @staticmethod
        def template_summary(findings, as_of):
            if not findings:
                return (f"Sentinel run for {as_of}: all checked series are inside their trailing "
                        "28-day control bands and the live schema matches the registry. No action needed.")
            by_sev = {s: 0 for s in sc.SEVERITIES}
            for f in findings:
                by_sev[f.severity] += 1
            counts = ", ".join(f"{by_sev[s]} {s}" for s in reversed(sc.SEVERITIES) if by_sev[s])
            worst = sc.worst_severity(findings) or "info"
            top = [f for f in findings if f.severity == worst][:3]
            out = [f"Sentinel run for {as_of} found {len(findings)} finding(s): {counts}."]
            if top:
                out.append(f"Top {worst} item(s): " + "; ".join(f.message.rstrip('.') for f in top) + ".")
            out.append("Affected check(s): " + ", ".join(sorted({f.check for f in findings})) + ".")
            out.append("Pending analyst approval before distribution, per the human-checkpoint design.")
            return " ".join(out)

        @staticmethod
        def narrate(findings, as_of, llm=None):
            if llm is not None and hasattr(llm, "chat_step"):
                try:
                    reply = llm.chat_step(
                        [{"role": "system", "content": "Narrate these sentinel findings in 3-5 sentences, using only the numbers given."},
                         {"role": "user", "content": json.dumps([f.to_dict() for f in findings])}],
                        tools=[],
                    )
                    content = (reply or {}).get("content")
                    if content and content.strip():
                        return content.strip()
                except Exception:
                    pass
            return sc.template_summary(findings, as_of)

    def get_llm():
        return None  # fallback mode: no agent.llm available -> template narration only

    print("Using the mirrored fallback sc / get_llm defined in this cell.")
else:
    print("Real import already active -- this fallback cell is a no-op.")

## 2. Read `sql/sentinel/*.sql` from the workspace

The exact same versioned files `scripts/sentinel.py` runs locally --
nothing is retyped here.

In [ ]:
SQL_DIR = f"{repo_root}/sql/sentinel"

def read_sql(name: str) -> str:
    with open(f"{SQL_DIR}/{name}", encoding="utf-8") as fh:
        return fh.read()

volume_sql_text = read_sql("daily_event_volumes.sql")
rates_sql_text = read_sql("daily_funnel_rates.sql")
schema_sql_text = read_sql("schema_snapshot.sql")
schema_statements = sc.load_named_statements(schema_sql_text)
print(f"Loaded {len(schema_statements)} named schema-snapshot statements: {list(schema_statements)}")

## 3. A tiny Spark-backed Queryable, and the registry

`sentinel_core.run_checks` only needs one method: `.query(sql) -> pandas`.
On Databricks that is `spark.sql(sql).toPandas()` -- everything else
(band math, registry diffing, narration) is identical to the CLI.

In [ ]:
class SparkQueryable:
    """agent.sentinel_core.Queryable adapter over a SparkSession -- the
    Databricks-side twin of agent.db.DatabricksDriver's query(), used only
    by this notebook so sentinel_core stays engine-agnostic."""

    def query(self, sql_text: str):
        return spark.sql(sql_text).toPandas()


driver = SparkQueryable()
registry = sc.load_registry(f"{repo_root}/config/sentinel_registry.json")
as_of = as_of_override or sc.default_as_of(driver)
print(f"Scoring as_of = {as_of}")

## 4. Run the three checks and collect findings

Read-only against `web_events` / `app_events` / `id_bridge` -- this Job
never writes to bronze/silver/gold or to the raw tables.

In [ ]:
volume_df = driver.query(sc.render_as_of(volume_sql_text, as_of))
rates_df = driver.query(sc.render_as_of(rates_sql_text, as_of))
columns_df = driver.query(schema_statements["columns"])
event_names_df = driver.query(schema_statements["event_names"])
app_versions_df = driver.query(schema_statements["app_versions"])

thresholds = registry["thresholds"]
findings = []
findings += sc.evaluate_volume_findings(volume_df, thresholds)
findings += sc.evaluate_funnel_rate_findings(rates_df, thresholds)
findings += sc.diff_columns(columns_df, registry)
findings += sc.diff_event_names(event_names_df, registry)
findings += sc.diff_app_versions(app_versions_df, registry)
findings += sc.check_missing_events_today(volume_df, registry, as_of)
findings = sc.sort_findings(findings)

worst = sc.worst_severity(findings)
code = sc.exit_code(findings)
print(f"{len(findings)} finding(s), worst severity = {worst}, exit_code = {code}")
for f in findings:
    print(f"  [{f.severity:>8}] {f.check}: {f.message}")

## 5. Narrate (LLM if configured, deterministic template otherwise) and write the DRAFT report

Same rule as the CLI: the LLM only phrases the findings above, it is never
asked to produce a number, and if no LLM is configured (or the call fails
for any reason -- no network, no key, ...) narration silently falls back
to the deterministic template. Either way the header below makes the
human checkpoint explicit: this Job does not notify anyone by itself.

In [ ]:
try:
    llm = get_llm()
except Exception:
    llm = None

narration = sc.narrate(findings, as_of, llm)
print(narration)

In [ ]:
import datetime as _dt

def render_markdown_report(as_of, findings, narration, volume_df, rates_df,
                            event_names_df, app_versions_df, columns_df, registry):
    """Mirrors scripts/sentinel.py's build_markdown_report -- kept small
    and inline here rather than imported, since it is presentation only
    (every NUMBER in it comes from the findings/registry objects computed
    by the real detection logic above, imported or mirrored)."""
    lines = [
        f"# Sentinel Report -- {as_of}", "",
        "**DRAFT -- pending analyst approval (human checkpoint)**", "",
        "## Executive Summary", "", narration, "",
        f"## Findings ({len(findings)})", "",
    ]
    for severity in ("critical", "warning", "info"):
        rows = [f for f in findings if f.severity == severity]
        lines.append(f"### {severity.capitalize()} ({len(rows)})")
        lines.append("")
        if not rows:
            lines.append("None.")
        else:
            lines += [f"- `{f.check}` **{f.subject}** -- {f.message}" for f in rows]
        lines.append("")
    lines += [
        "## Schema Snapshot", "",
        f"- Distinct event_name values observed: {sorted(event_names_df['event_name'].unique().tolist())}",
        f"- Distinct app_version values observed: {sorted(app_versions_df['app_version'].unique().tolist())}",
        "", "## Run Metadata", "",
        f"- as_of: {as_of}",
        f"- driver: databricks (SparkQueryable)",
        f"- exit_code: {sc.exit_code(findings)}",
        f"- registry generated_at: {registry.get('generated_at', 'unknown')}",
        f"- generated_at (this run, UTC): {_dt.datetime.now(_dt.timezone.utc).isoformat()}",
        "",
    ]
    return "\n".join(lines)

report_md = render_markdown_report(as_of, findings, narration, volume_df, rates_df,
                                    event_names_df, app_versions_df, columns_df, registry)

volume_path = f"/Volumes/{catalog}/{schema}/raw/sentinel"
dbutils.fs.mkdirs(volume_path)
report_path = f"{volume_path}/sentinel_{as_of}.md"
dbutils.fs.put(report_path, report_md, overwrite=True)
print(f"Report written to {report_path}")

## 6. Job-friendly exit

`dbutils.notebook.exit` surfaces the run's severity to a Databricks
Workflows Job so a downstream task (or the Job's own alerting) can branch
on it, without this notebook ever sending a notification itself -- exactly
the `--notify` behaviour of the CLI (prints/records intent, never sends).

In [ ]:
result = {
    "as_of": as_of,
    "exit_code": sc.exit_code(findings),
    "worst_severity": sc.worst_severity(findings),
    "finding_count": len(findings),
    "report_path": report_path,
}
print(result)
dbutils.notebook.exit(json.dumps(result))